# This is not how to write a clean ETL code this is just a short code to read.

In [0]:
df = spark.read.csv("/Volumes/databricks_lakehouse/bronze/source_systems/source_crm/cust_info.csv",
                    header=True,
                    inferSchema=True
                    )
display(df)


# This is how to read properply cause it is easy to read and can be customized anytime much easily.This is kinda the ETL standard.

In [0]:
df = (
    spark.read.option("header", True)
    .option("inferSchema", True)
    .format("csv")    # or just use .csv("path") instead of 2 lines.
    .load("/Volumes/databricks_lakehouse/bronze/source_systems/source_crm/cust_info.csv")
)
# we can also write without the () by just putting \ at the end of each line to break them for easier reading
# like .option("header", True) \
# .option("inferSchema", True) \
display(df)

# Now we write it into a delta table.

In [0]:
df.write.mode("overwrite").saveAsTable("databricks_lakehouse.bronze.crm_cust_info")

# Now as we can see we have 6 such files and we should not do this one by one.

# So we make an Ingestion Configuration file with all the info.

In [0]:
INGESTION_CONFIG = [
    {
        "source": "crm",
        "path": "/Volumes/databricks_lakehouse/bronze/source_systems/source_crm/cust_info.csv",
        "table": "crm_cust_info"
    },
    {
        "source": "crm",
        "path": "/Volumes/databricks_lakehouse/bronze/source_systems/source_crm/prd_info.csv",
        "table": "crm_prd_info"
    },
    {
        "source": "crm",
        "path": "/Volumes/databricks_lakehouse/bronze/source_systems/source_crm/sales_details.csv",
        "table": "crm_sales_details"
    },
    {
        "source": "erp",
        "path": "/Volumes/databricks_lakehouse/bronze/source_systems/source_erp/CUST_AZ12.csv",
        "table": "erp_CUST_AZ12"
    },
    {
        "source": "erp",
        "path": "/Volumes/databricks_lakehouse/bronze/source_systems/source_erp/LOC_A101.csv",
        "table": "erp_LOC_A101"
    },
    {
        "source": "erp",
        "path": "/Volumes/databricks_lakehouse/bronze/source_systems/source_erp/PX_CAT_G1V2.csv",
        "table": "erp_PX_CAT_G1V2"
    }
]

#Now we Ingest files into bronze layer

In [0]:
for item in INGESTION_CONFIG:

    print(f"Ingesting {item['source']} → workspace.bronze.{item['table']}")

    df = (
        spark.read.option("header", True)
        .option("inferSchema", True)
        .format("csv")
        .load(item["path"])
    )

    (
        df.write
        .mode("overwrite")
        .format("delta")  # here I have explicitly written the format as delta, which is the default format.
        .saveAsTable(f"databricks_lakehouse.bronze.{item['table']}")
    )